# Excited-State Fixture Tables (DALTON + MADNESS)

This notebook reads excited-state fixture data and builds pandas tables.

- DALTON: `dalton_fixtures/h2-excited-state` via `gecko.load_calc`
- MADNESS: `*.calc_info.ref.json` (calc_json fixtures) from `src/apps/madqc_v2`

Each table includes `basis` and `molecule` columns for each calculation.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd

try:
    import gecko
except ImportError:
    import sys

    gecko_src = Path('/gpfs/projects/rjh/adrian/development/gecko/src')
    if gecko_src.exists():
        sys.path.insert(0, str(gecko_src))
    import gecko

from gecko.plugins.madness.parse import _extract_excited_states, _infer_mra_basis_from_obj



In [2]:
DALTON_FIXTURE_ROOT = Path('/gpfs/projects/rjh/adrian/development/madness-worktrees/molresponse-feature-next/dalton_fixtures/h2-excited-state')
MADNESS_CALC_JSON_ROOT = Path('/gpfs/projects/rjh/adrian/development/madness-worktrees/molresponse-feature-next/src/apps/madqc_v2')
MADNESS_RUNTIME_CALC_JSONS = [
    Path('/gpfs/projects/rjh/adrian/development/madness-worktrees/molresponse-feature-next/analysis/results/h2_excited_daug_cc_pvqz_protocol_1e-4_1e-6/h2_excited_mra.calc_info.json'),
]

HARTREE_TO_EV = 27.211386245988


def _formula_from_symbols(symbols: list[str]) -> str:
    counts = Counter(symbols)
    order = []
    for sym in symbols:
        if sym not in order:
            order.append(sym)
    return ''.join(f"{sym}{counts[sym] if counts[sym] > 1 else ''}" for sym in order)


def _formula_from_qcel_molecule(mol) -> str | None:
    if mol is None:
        return None
    symbols = getattr(mol, 'symbols', None)
    if symbols is None:
        return None
    return _formula_from_symbols([str(sym) for sym in symbols])


def _infer_molecule_from_name(path: Path) -> str:
    token_map = {
        'h2': 'H2',
        'he': 'He',
        'h2o': 'H2O',
        'lih': 'LiH',
        'n2': 'N2',
    }
    tokens = [tok for tok in re.split(r'[^a-zA-Z0-9]+', path.stem.lower()) if tok]
    for token in tokens:
        if token in token_map:
            return token_map[token]
    return path.stem


def _first_molecule_formula_from_calc_json(raw: dict) -> str | None:
    tasks = raw.get('tasks')
    if not isinstance(tasks, list):
        return None
    for task in tasks:
        if not isinstance(task, dict):
            continue
        mol = task.get('molecule')
        if not isinstance(mol, dict):
            continue
        symbols = mol.get('symbols')
        if isinstance(symbols, list) and symbols:
            return _formula_from_symbols([str(sym) for sym in symbols])
    return None


def _extract_response_excited_bundle_rows(raw_json: dict) -> list[dict]:
    tasks = raw_json.get('tasks')
    if not isinstance(tasks, list):
        return []

    rows: list[dict] = []
    for task_index, task in enumerate(tasks):
        if not isinstance(task, dict):
            continue
        if task.get('type') != 'response':
            continue

        metadata = task.get('metadata')
        if not isinstance(metadata, dict):
            continue

        excited = metadata.get('excited_states')
        if not isinstance(excited, dict):
            continue

        plan = excited.get('plan') if isinstance(excited.get('plan'), dict) else {}
        protocols = excited.get('protocols')
        if not isinstance(protocols, dict):
            continue

        for protocol_key, protocol_data in sorted(protocols.items()):
            if not isinstance(protocol_data, dict):
                continue
            energies = protocol_data.get('energies')
            if not isinstance(energies, list):
                continue

            for excitation_index, omega in enumerate(energies):
                try:
                    omega_au = float(omega)
                except (TypeError, ValueError):
                    continue

                rows.append({
                    'task_index': task_index,
                    'task_model': str(task.get('model')) if task.get('model') is not None else None,
                    'task_type': str(task.get('type')) if task.get('type') is not None else None,
                    'excitation_index': excitation_index,
                    'mode': excitation_index + 1,
                    'symmetry_excited_state': None,
                    'omega_au': omega_au,
                    'omega_ev': omega_au * HARTREE_TO_EV,
                    'protocol_key': protocol_key,
                    'protocol_converged': protocol_data.get('converged'),
                    'protocol_saved': protocol_data.get('saved'),
                    'protocol_iterations': protocol_data.get('iterations'),
                    'protocol_stage_status': protocol_data.get('stage_status'),
                    'plan_num_states': plan.get('num_states'),
                    'plan_protocols': plan.get('protocols'),
                    'source_kind': 'response_excited_bundle',
                })

    return rows



In [3]:
dalton_calc = gecko.load_calc(DALTON_FIXTURE_ROOT)
dalton_rows = []

for out_name, out_data in sorted((dalton_calc.data.get('dalton_outputs') or {}).items()):
    basis = out_data.get('basis') or dalton_calc.meta.get('basis')
    molecule = (
        _formula_from_qcel_molecule(out_data.get('molecule'))
        or _formula_from_qcel_molecule(dalton_calc.molecule)
        or out_data.get('label')
        or dalton_calc.meta.get('molecule')
        or 'unknown'
    )
    for row in out_data.get('excited_states') or []:
        dalton_rows.append({
            'code': 'dalton',
            'calculation': out_name,
            'basis': basis,
            'molecule': molecule,
            **row,
        })

dalton_df = pd.DataFrame(dalton_rows)
if not dalton_df.empty:
    dalton_df = dalton_df.sort_values(['calculation', 'symmetry_excited_state', 'mode']).reset_index(drop=True)

dalton_df



,code,calculation,basis,molecule,spin,symmetry_excited_state,mode,omega_au,omega_ev
0,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,1,1,0.479814,13.056405
1,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,1,2,0.742497,20.204380
2,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,1,3,0.985336,26.812355
3,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,1,4,2.208070,60.084641
4,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,2,1,0.577321,15.709695
...,...,...,...,...,...,...,...,...,...
103,dalton,excited_H2-d-aug-cc-pVQZ.out,d-aug-cc-pVQZ,H2,singlet,7,4,0.901287,24.525261
104,dalton,excited_H2-d-aug-cc-pVQZ.out,d-aug-cc-pVQZ,H2,singlet,8,1,0.647379,17.616084
105,dalton,excited_H2-d-aug-cc-pVQZ.out,d-aug-cc-pVQZ,H2,singlet,8,2,0.895877,24.378045
106,dalton,excited_H2-d-aug-cc-pVQZ.out,d-aug-cc-pVQZ,H2,singlet,8,3,1.398861,38.064944


In [4]:
calc_json_paths = sorted(MADNESS_CALC_JSON_ROOT.glob('mad_madqc_test_*.calc_info.ref.json'))
calc_json_paths.extend(path for path in MADNESS_RUNTIME_CALC_JSONS if path.exists())
calc_json_paths = sorted(set(calc_json_paths))

madness_rows = []
for calc_json_path in calc_json_paths:
    raw = json.loads(calc_json_path.read_text(encoding='utf-8'))

    task_excited_rows = _extract_excited_states(raw)
    response_bundle_rows = _extract_response_excited_bundle_rows(raw)
    extracted_rows = task_excited_rows + response_bundle_rows
    if not extracted_rows:
        continue

    basis = _infer_mra_basis_from_obj(raw) or 'mra'
    molecule = _first_molecule_formula_from_calc_json(raw) or _infer_molecule_from_name(calc_json_path)

    for row in extracted_rows:
        source_kind = row.get('source_kind', 'task_excitations')
        madness_rows.append({
            'code': 'madness',
            'calculation': calc_json_path.name,
            'basis': basis,
            'molecule': molecule,
            'source_kind': source_kind,
            **row,
        })

madness_df = pd.DataFrame(madness_rows)
if not madness_df.empty:
    sort_cols = [
        col for col in ['calculation', 'source_kind', 'protocol_key', 'task_index', 'excitation_index']
        if col in madness_df.columns
    ]
    madness_df = madness_df.sort_values(sort_cols).reset_index(drop=True)

madness_df



,code,calculation,basis,molecule,source_kind,task_index,task_model,task_type,task_nfreeze,excitation_index,current_error,irrep,omega,oscillator_strength_length,oscillator_strength_velocity,omega_au,omega_ev
0,madness,mad_madqc_test_cis_energy_he.py.calc_info.ref....,mra,He,task_excitations,1,cis,None,0,0,0.425231,a,0.851804,1.856574e-01,1.437783e-01,0.851804,23.178775
1,madness,mad_madqc_test_cis_energy_he.py.calc_info.ref....,mra,He,task_excitations,1,cis,None,0,1,0.779312,a,0.859847,2.115104e-21,9.599893e-21,0.859847,23.397617
2,madness,mad_madqc_test_cis_symmetry_h2o.py_a2.calc_inf...,mra,H2O,task_excitations,1,cis,None,1,0,0.000595,a2,0.384218,1.136654e-25,4.669598e-27,0.384218,10.455114
3,madness,mad_madqc_test_cis_symmetry_h2o.py_b1.calc_inf...,mra-d04,H2O,task_excitations,1,cis,None,1,0,0.001299,b1,0.323416,4.860732e-02,6.652912e-02,0.323416,8.800609
4,madness,mad_madqc_test_lrcc2_helium.py.calc_info.ref.json,mra-d03,mad_madqc_test_lrcc2_helium.py.calc_info.ref,task_excitations,1,cc2,None,0,0,0.000000,,0.765272,3.000000e+00,0.000000e+00,0.765272,20.824109


In [5]:
all_excited_df = pd.concat([dalton_df, madness_df], ignore_index=True, sort=False)

summary_df = (
    all_excited_df.groupby(['code', 'calculation', 'molecule', 'basis'], dropna=False)
    .size()
    .rename('n_excitations')
    .reset_index()
    .sort_values(['code', 'calculation'])
    .reset_index(drop=True)
)

summary_df



,code,calculation,molecule,basis,n_excitations
0,dalton,excited_H2-aug-cc-pVDZ.out,H2,aug-cc-pVDZ,16
1,dalton,excited_H2-aug-cc-pVQZ.out,H2,aug-cc-pVQZ,32
2,dalton,excited_H2-aug-cc-pVTZ.out,H2,aug-cc-pVTZ,28
3,dalton,excited_H2-d-aug-cc-pVQZ.out,H2,d-aug-cc-pVQZ,32
4,madness,mad_madqc_test_cis_energy_he.py.calc_info.ref....,He,mra,2
5,madness,mad_madqc_test_cis_symmetry_h2o.py_a2.calc_inf...,H2O,mra,1
6,madness,mad_madqc_test_cis_symmetry_h2o.py_b1.calc_inf...,H2O,mra-d04,1
7,madness,mad_madqc_test_lrcc2_helium.py.calc_info.ref.json,mad_madqc_test_lrcc2_helium.py.calc_info.ref,mra-d03,1


In [6]:
all_excited_df



,code,calculation,basis,molecule,spin,symmetry_excited_state,mode,omega_au,omega_ev,source_kind,task_index,task_model,task_type,task_nfreeze,excitation_index,current_error,irrep,omega,oscillator_strength_length,oscillator_strength_velocity
0,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,1.0,1.0,0.479814,13.056405,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,1.0,2.0,0.742497,20.204380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,1.0,3.0,0.985336,26.812355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,1.0,4.0,2.208070,60.084641,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,dalton,excited_H2-aug-cc-pVDZ.out,aug-cc-pVDZ,H2,singlet,2.0,1.0,0.577321,15.709695,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,madness,mad_madqc_test_cis_energy_he.py.calc_info.ref....,mra,He,NaN,NaN,NaN,0.851804,23.178775,task_excitations,1.0,cis,None,0.0,0.0,0.425231,a,0.851804,1.856574e-01,1.437783e-01
109,madness,mad_madqc_test_cis_energy_he.py.calc_info.ref....,mra,He,NaN,NaN,NaN,0.859847,23.397617,task_excitations,1.0,cis,None,0.0,1.0,0.779312,a,0.859847,2.115104e-21,9.599893e-21
110,madness,mad_madqc_test_cis_symmetry_h2o.py_a2.calc_inf...,mra,H2O,NaN,NaN,NaN,0.384218,10.455114,task_excitations,1.0,cis,None,1.0,0.0,0.000595,a2,0.384218,1.136654e-25,4.669598e-27
111,madness,mad_madqc_test_cis_symmetry_h2o.py_b1.calc_inf...,mra-d04,H2O,NaN,NaN,NaN,0.323416,8.800609,task_excitations,1.0,cis,None,1.0,0.0,0.001299,b1,0.323416,4.860732e-02,6.652912e-02


In [7]:
TARGET_DALTON_CALC = 'excited_H2-d-aug-cc-pVQZ.out'
TARGET_MADNESS_CALC = 'v4.calc_info.json'
TARGET_MADNESS_PROTOCOL = '1e-06'

dalton_target_df = (
    dalton_df[(dalton_df['calculation'] == TARGET_DALTON_CALC) & (dalton_df['molecule'] == 'H2')]
    .sort_values('omega_au')
    .reset_index(drop=True)
)
dalton_target_df['state_rank'] = dalton_target_df.index + 1

madness_target_df = madness_df[(madness_df['calculation'] == TARGET_MADNESS_CALC) & (madness_df['molecule'] == 'H2')].copy()
if 'source_kind' in madness_target_df.columns:
    madness_target_df = madness_target_df[madness_target_df['source_kind'] == 'response_excited_bundle']
if 'protocol_key' in madness_target_df.columns:
    madness_target_df = madness_target_df[madness_target_df['protocol_key'] == TARGET_MADNESS_PROTOCOL]
madness_target_df = madness_target_df.sort_values('omega_au').reset_index(drop=True)
madness_target_df['state_rank'] = madness_target_df.index + 1

requested_states = None
if 'plan_num_states' in madness_target_df.columns and not madness_target_df['plan_num_states'].dropna().empty:
    requested_states = int(madness_target_df['plan_num_states'].dropna().iloc[0])

protocol_converged = []
if 'protocol_converged' in madness_target_df.columns:
    protocol_converged = sorted(set(madness_target_df['protocol_converged'].dropna().tolist()))

comparison_meta_df = pd.DataFrame(
    [
        {'metric': 'dalton_states_d-aug-cc-pVQZ', 'value': int(len(dalton_target_df))},
        {'metric': 'madness_requested_states', 'value': requested_states},
        {'metric': 'madness_states_protocol_1e-06', 'value': int(len(madness_target_df))},
        {'metric': 'madness_protocol_converged_flags', 'value': str(protocol_converged)},
    ]
)

comparison_df = (
    dalton_target_df[['state_rank', 'omega_au', 'omega_ev']]
    .rename(columns={'omega_au': 'dalton_omega_au', 'omega_ev': 'dalton_omega_ev'})
    .merge(
        madness_target_df[
            [col for col in ['state_rank', 'omega_au', 'omega_ev', 'protocol_stage_status'] if col in madness_target_df.columns]
        ]
        .rename(columns={'omega_au': 'madness_omega_au', 'omega_ev': 'madness_omega_ev'}),
        on='state_rank',
        how='outer',
    )
    .sort_values('state_rank')
    .reset_index(drop=True)
)

comparison_df['delta_au'] = comparison_df['madness_omega_au'] - comparison_df['dalton_omega_au']
comparison_df['delta_ev'] = comparison_df['madness_omega_ev'] - comparison_df['dalton_omega_ev']

display(comparison_meta_df)
comparison_df



,metric,value
0,dalton_states_d-aug-cc-pVQZ,32
1,madness_requested_states,None
2,madness_states_protocol_1e-06,0
3,madness_protocol_converged_flags,[]


,state_rank,dalton_omega_au,dalton_omega_ev,madness_omega_au,madness_omega_ev,delta_au,delta_ev
0,1,0.465376,12.663536,NaN,NaN,NaN,NaN
1,2,0.477069,12.981695,NaN,NaN,NaN,NaN
2,3,0.480889,13.085666,NaN,NaN,NaN,NaN
3,4,0.480889,13.085666,NaN,NaN,NaN,NaN
4,5,0.537139,14.616300,NaN,NaN,NaN,NaN
5,6,0.541781,14.742612,NaN,NaN,NaN,NaN
6,7,0.544758,14.823610,NaN,NaN,NaN,NaN
7,8,0.544758,14.823610,NaN,NaN,NaN,NaN
8,9,0.547130,14.888172,NaN,NaN,NaN,NaN
9,10,0.573313,15.600630,NaN,NaN,NaN,NaN
